In [2]:
# Ejecutar solo si faltan paquetes (evitar actualizar pandas dentro del notebook: consume RAM)
# %pip install numpy pandas scikit-learn

In [ ]:
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# =============================================================================
# 1. Configuración de Rutas
# =============================================================================
RUTA_CLIMA = Path("../Datos/Staging/clima_limpio.csv")
RUTA_ACCIDENTES = Path("../Datos/Raw/accidentes.csv")
RUTA_RAW_STAGING = Path("../Datos/Staging/accidentes_staging_enriquecido.csv")
RUTA_TRUSTED = Path("../Datos/Trusted/clima_accidentes_modelo.csv")

TARGET_COL = "hubo_accidente"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CHUNK_SIZE = 200_000  # chunks pequeños para no saturar RAM
COLS_DROPNA = ["temperature", "humidity", "windSpeed"]

# =============================================================================
# 2. Fase 1: solo el vector target (~7 MB) para split estratificado
#    Mismo filtro dropna que en Fase 3 → los índices coinciden
# =============================================================================
print("--> Fase 1: Construyendo vector target para el split (bajo uso de RAM)...")

if RUTA_TRUSTED.exists():
    print(f"   Usando target desde {RUTA_TRUSTED.name} (con mismo dropna que Fase 3)")
    y_partes = []
    for chunk in pd.read_csv(
        RUTA_TRUSTED,
        usecols=["target", *COLS_DROPNA],
        chunksize=CHUNK_SIZE,
        dtype={"target": "uint8", "temperature": "float32", "humidity": "float32", "windSpeed": "float32"},
    ):
        chunk = chunk.dropna(subset=COLS_DROPNA)
        y_partes.append(chunk["target"].to_numpy())
        del chunk
        gc.collect()
    y_all = np.concatenate(y_partes)
    del y_partes
else:
    print("   Trusted no encontrado: calculando target por merge con accidentes...")
    acc_keys = (
        pd.read_csv(RUTA_ACCIDENTES, usecols=["BARRIO", "TW"], parse_dates=["TW"])
        .drop_duplicates()
    )
    cols_lectura = ["BARRIO", "TW", *COLS_DROPNA]
    y_partes = []
    for chunk in pd.read_csv(
        RUTA_CLIMA,
        usecols=cols_lectura,
        parse_dates=["TW"],
        chunksize=CHUNK_SIZE,
        dtype={
            "temperature": "float32",
            "humidity": "float32",
            "windSpeed": "float32",
        },
    ):
        chunk = chunk.dropna(subset=COLS_DROPNA)
        merged = chunk.merge(acc_keys, on=["BARRIO", "TW"], how="left", indicator=True)
        y_partes.append((merged["_merge"] == "both").astype(np.uint8).to_numpy())
        del chunk, merged
        gc.collect()
    y_all = np.concatenate(y_partes)
    del y_partes, acc_keys
    gc.collect()

total_c = len(y_all)
pos_c = int(y_all.sum())
print(f"   Universo detectado: {total_c:,} filas.")
print(f"   [RESPUESTA TALLER]: Proporción global de clase positiva: {(pos_c / total_c) * 100:.4f}%")

indices = np.arange(total_c, dtype=np.int64)
indices_train, indices_test = train_test_split(
    indices,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE,
)
del indices, y_all, indices_test
gc.collect()

# =============================================================================
# 3. Carga de tu Feature Engineering desde Staging
# =============================================================================
print("\n--> Fase 2: Cargando tus variables enriquecidas...")
raw_staging = pd.read_csv(RUTA_RAW_STAGING, parse_dates=["TW"])

columnas_features_agg = [
    "hora_dia", "dia_semana", "mes_num", "es_fin_semana", "es_festivo", 
    "hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos", "dia_ano_sin", 
    "dia_ano_cos", "comuna_ext", "prom_accidentes_historico_barrio"
]

agg_dict = {"Lat": "median", "Lon": "median"}
for col in columnas_features_agg:
    if col in raw_staging.columns:
        agg_dict[col] = "first"

features_agregadas = raw_staging.groupby(["BARRIO", "TW"]).agg(agg_dict).reset_index()
features_agregadas["BARRIO"] = features_agregadas["BARRIO"].astype("category")
features_agregadas["Lat"] = features_agregadas["Lat"].astype("float32")
features_agregadas["Lon"] = features_agregadas["Lon"].astype("float32")
del raw_staging

# =============================================================================
# 4. Paso 2: Re-procesar por bloques distribuyendo directamente a Train y Test
# =============================================================================
print("\n--> Fase 3: Construyendo matrices X e y finales separadas en RAM...")

DTYPES_CLIMA = {
    "BARRIO": "category", "temperature": "float32", "apparentTemperature": "float32",
    "humidity": "float32", "precipIntensity": "float32", "precipProbability": "float32",
    "windSpeed": "float32", "cloudCover": "float32", "visibility": "float32"
}

X_train_partes, y_train_partes = [], []
X_test_partes, y_test_partes = [], []

fila_actual_global = 0

acc_keys = (
    pd.read_csv(RUTA_ACCIDENTES, usecols=["BARRIO", "TW"], parse_dates=["TW"])
    .drop_duplicates()
)

# Códigos de barrio consistentes en todos los chunks
barrios_unicos = []
for _c in pd.read_csv(RUTA_CLIMA, usecols=["BARRIO"], chunksize=CHUNK_SIZE):
    barrios_unicos.append(_c["BARRIO"].astype(str).unique())
BARRIO_MAP = {b: i for i, b in enumerate(pd.unique(np.concatenate(barrios_unicos)))}
del barrios_unicos

for chunk in pd.read_csv(RUTA_CLIMA, chunksize=CHUNK_SIZE, parse_dates=["TW"], dtype=DTYPES_CLIMA):
    chunk = chunk.dropna(subset=COLS_DROPNA)
    chunk["BARRIO"] = chunk["BARRIO"].astype("category")
    
    # Pegamos tus variables enriquecidas
    chunk_enriquecido = pd.merge(chunk, features_agregadas, on=["BARRIO", "TW"], how="left")
    
    # Target por merge (sin strings gigantes ni set en RAM)
    chunk_enriquecido = chunk_enriquecido.merge(
        acc_keys.assign(**{TARGET_COL: 1}),
        on=["BARRIO", "TW"],
        how="left",
    )
    chunk_enriquecido[TARGET_COL] = chunk_enriquecido[TARGET_COL].fillna(0).astype(np.uint8)
    
    # Índices alineados con Fase 1 (mismo dropna y orden de lectura)
    indices_del_chunk = np.arange(
        fila_actual_global, fila_actual_global + len(chunk_enriquecido), dtype=np.int64
    )
    fila_actual_global += len(chunk_enriquecido)
    
    es_train = np.isin(indices_del_chunk, indices_train)
    
    # Limpieza final de columnas predictoras
    if "BARRIO" in chunk_enriquecido.columns:
        chunk_enriquecido["BARRIO"] = (
            chunk_enriquecido["BARRIO"].astype(str).map(BARRIO_MAP).fillna(-1).astype(np.int32)
        )
    
    chunk_predictores = chunk_enriquecido.drop(columns=["TW", "summary", "icon", "fecha_dt", TARGET_COL], errors="ignore").fillna(0)
    chunk_target = chunk_enriquecido[TARGET_COL]
    
    # Separación inmediata sin acumular en una sola lista gigante
    X_train_partes.append(chunk_predictores[es_train])
    y_train_partes.append(chunk_target[es_train])
    X_test_partes.append(chunk_predictores[~es_train])
    y_test_partes.append(chunk_target[~es_train])

del features_agregadas, acc_keys, indices_train, BARRIO_MAP
gc.collect()

# Concatemos los conjuntos por separado (ocupan la mitad de espacio que la tabla completa unificada)
print("--> Consolidando conjuntos de datos finales...")
X_train = pd.concat(X_train_partes, ignore_index=True)
y_train = pd.concat(y_train_partes, ignore_index=True)
X_test = pd.concat(X_test_partes, ignore_index=True)
y_test = pd.concat(y_test_partes, ignore_index=True)

del X_train_partes, y_train_partes, X_test_partes, y_test_partes
print(f"¡Split seguro completado! Registros Entrenamiento: {len(X_train):,}, Registros Prueba: {len(X_test):,}")

# =============================================================================
# 5. Reporte de Distribución de Clases (Código del Profesor)
# =============================================================================
def get_binary_class_counts(split_y):
    neg_c = int((split_y == 0).sum())
    pos_c = int((split_y == 1).sum())
    total = len(split_y)
    return {"negative_count": neg_c, "positive_count": pos_c, "positive_rate": pos_c / total if total > 0 else 0}

split_rows = []
for split_name, split_y in {"train": y_train, "test": y_test}.items():
    stats = get_binary_class_counts(split_y)
    split_rows.append({"split": split_name, **stats, "positive_rate_percent": stats["positive_rate"] * 100})

try:
    display(pd.DataFrame(split_rows))
except NameError:
    print(pd.DataFrame(split_rows))

--> Fase 1: Construyendo vector target para el split (bajo uso de RAM)...
   Trusted no encontrado: calculando target por merge con accidentes...
   Universo detectado: 7,436,188 filas.
   [RESPUESTA TALLER]: Proporción global de clase positiva: 1.5139%

--> Fase 2: Cargando tus variables enriquecidas...


C:\Users\camil\AppData\Local\Temp\ipykernel_18764\838741493.py:89: DtypeWarning: Columns (0: RADICADO, 1: CBML) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_staging = pd.read_csv(RUTA_RAW_STAGING, parse_dates=["TW"])



--> Fase 3: Construyendo matrices X e y finales separadas en RAM...
--> Consolidando conjuntos de datos finales...
¡Split seguro completado! Registros Entrenamiento: 5,948,950, Registros Prueba: 1,487,238


,split,negative_count,positive_count,positive_rate,positive_rate_percent
0,train,5858890,90060,0.015139,1.513881
1,test,1464723,22515,0.015139,1.513880
